# AI Chatbot using ML APIs
Run the cells one by one with Shift+Enter. Keep this notebook in the same folder as `data.csv` and `responses.py`.

## 1. Create dataset (run once)

In [ ]:
%run generate_data.py

## 2. Load and explore data (EDA)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv('data.csv')
print(df.head())
print('Missing values:', df.isnull().sum().sum())
df['intent'].value_counts().plot(kind='barh', figsize=(6,6), title='Sentences per intent')
plt.show()

## 3. Clean, convert text to numbers, split

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
df = df.dropna()
df['text'] = df['text'].str.lower().str.strip()
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X = vectorizer.fit_transform(df['text'])
y = df['intent']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape[0], 'Test:', X_test.shape[0])

## 4. Train and evaluate

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
model = LogisticRegression(C=10, max_iter=1000)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))
fig, ax = plt.subplots(figsize=(9,9))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred, labels=model.classes_), display_labels=model.classes_).plot(ax=ax, xticks_rotation=90)
plt.show()

## 5. Save final model (trained on all data)

In [ ]:
import pickle
model.fit(X, y)
pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(vectorizer, open('vectorizer.pkl', 'wb'))
print('Saved!')

## 6. Chat
Optional: set your API key in the next cell to answer any topic.

In [ ]:
import os
# os.environ['GEMINI_API_KEY'] = 'PASTE_YOUR_KEY_HERE'   # remove the # and paste key
import chatbot
print('API mode:', 'ON' if chatbot.client else 'OFF')

In [ ]:
while True:
    user = input('You: ').strip()
    if user.lower() in ('quit', 'exit'):
        print('Bot: Bye!')
        break
    if user:
        source, reply, conf = chatbot.get_reply(user)
        print(f'Bot [{source}, {conf:.2f}]: {reply}')